# task_09 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_09/data/")


In [2]:
campaigns = pd.read_csv(base / "campaigns.csv")
ad_spend = pd.read_csv(base / "ad_spend.csv")
sessions = pd.read_csv(base / "sessions.csv")
orders = pd.read_csv(base / "orders.csv")

Q1: Using revenue attributed only to orders linked to sessions in this dataset, which channel has the highest return on ad spend?

In [3]:
sessions["session_date"] = pd.to_datetime(sessions["session_date"])
ad_spend["date"] = pd.to_datetime(ad_spend["date"])

sessions_orders = sessions.merge(orders, on="session_id").merge(campaigns, on="campaign_id")
revenue_by_channel = sessions_orders.groupby("channel")["order_value"].sum()
spend_by_channel = ad_spend.merge(campaigns, on="campaign_id").groupby("channel")["spend"].sum()
q1 = str((revenue_by_channel / spend_by_channel).sort_values(ascending=False).index[0])
print(q1)


display


Q2: Among search-channel sessions, which is the earliest session_date with the highest bounce rate, where bounce means pages_viewed = 1?

In [4]:
search_sessions = sessions.merge(campaigns[["campaign_id", "channel"]], on="campaign_id")
search_sessions = search_sessions[search_sessions["channel"] == "search"]
bounce_rate = search_sessions.groupby("session_date").apply(lambda g: (g["pages_viewed"] == 1).mean(), include_groups=False)
q2 = bounce_rate[bounce_rate == bounce_rate.max()].index.min().date().isoformat()
print(q2)


2024-03-04


Q4: How many users had sessions from multiple channels before their first purchase?

In [5]:
sessions_with_channel = sessions.merge(campaigns[["campaign_id", "channel"]], on="campaign_id")
purchase_sessions = sessions.merge(orders[["session_id"]], on="session_id")
first_purchase_date = purchase_sessions.groupby("user_id")["session_date"].min()
users_multi_channel = 0
for user_id, first_date in first_purchase_date.items():
    channels_before = sessions_with_channel[(sessions_with_channel["user_id"] == user_id) & (sessions_with_channel["session_date"] < first_date)]["channel"].nunique()
    if channels_before >= 2:
        users_multi_channel += 1
q4 = str(users_multi_channel)
print(q4)


10
